# holdout-data-one-per-class — ex1: select one sample per class for the holdout gallery

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `holdout-data-one-per-class`. Running the final beacon cell reports progress against the `Generative: Hold-out one-per-class data` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: Hold-out one-per-class data` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`holdout-data-one-per-class`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "holdout-data-one-per-class"
DD_SUBTOPIC = "Generative: Hold-out one-per-class data"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Hold-out one-per-class data — quick refresher

When logging autoencoder / VAE reconstructions during training you want a FIXED, REPRESENTATIVE batch — usually one image per class — so the visualization tells the same story across epochs.

The pattern:
```python
holdout = []
for c in range(num_classes):
    mask = labels == c
    holdout.append(data[mask][0])    # FIRST sample of class c
holdout = t.stack(holdout, dim=0)    # (num_classes, *sample_shape)
```

**Why ONE per class.** Cheaper to log, easier to read on a 1×K grid, and guarantees every class is represented even if the test set is imbalanced. The first hit of each class is fine — you don't need a random pick because the dataset shuffle is upstream.

**Mask indexing returns a copy.** `data[mask]` always allocates fresh storage. Take `[0]` rather than `[:1]` if you want a non-batch shape; the final `t.stack` re-adds the leading axis.

### Exercise 1 — select one sample per class for the holdout gallery

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply boolean-mask selection (`labels == c`) inside a per-class loop to extract the FIRST sample of every class, then stack into a `(num_classes, *sample_shape)` holdout tensor.
> Keywords: holdout, boolean-mask, stack, gallery
> ```

**KCs targeted:** `mask-equals-class`, `stack-per-class-firsts`

Implement `ex1_one_per_class(data, labels, num_classes)`. The logging-batch construction pattern that ARENA uses for both autoencoder and VAE training:

1. `data` has shape `(N, *sample_shape)`. `labels` is a 1-D `int64` tensor of length `N`, each in `[0, num_classes)`. They are in matched order.
2. For each class `c` in `0..num_classes-1`:
   - Build `mask = labels == c` (a `(N,)` bool tensor).
   - Select `data[mask][0]` — the FIRST sample whose label is `c`.
3. Stack the resulting list with `t.stack(per_class, dim=0)`.
4. Return the `(num_classes, *sample_shape)` tensor.

Assume every class appears at least once in the batch — no need to handle empty masks.

Input: `(N, *)` data, `(N,)` int64 labels, int `num_classes`.
Output: `(num_classes, *)` tensor, indexed by class.

The visualization renders the gallery as a 1×K grid so you can visually verify each row holds one sample of the corresponding class.

In [ ]:
def ex1_one_per_class(data: Tensor, labels: Tensor, num_classes: int) -> Tensor:
    per_class = []
    for c in range(num_classes):
        mask = labels == c
        per_class.append(data[mask][0])
    return t.stack(per_class, dim=0)


<details><summary>Solution</summary>

```python
def ex1_one_per_class(data: Tensor, labels: Tensor, num_classes: int) -> Tensor:
    per_class = []
    for c in range(num_classes):
        mask = labels == c
        per_class.append(data[mask][0])
    return t.stack(per_class, dim=0)
```

**`data[mask]` returns a copy.** Boolean-mask indexing always allocates fresh storage — there's no view-version that does it. That's fine here (holdout is tiny), but if you ever boolean-mask inside a tight inner loop you'll spend time profiling allocator pressure.

**Why `data[mask][0]` not `data[mask][:1]`.** Both work. `[0]` returns the sample with the leading batch axis removed (shape `(*sample_shape,)`); `[:1]` keeps the axis. `t.stack(..., dim=0)` re-adds the leading axis regardless — but starting from the axis-removed form makes the stacking semantics symmetric across tabular and image cases.

**Failure mode this would catch.** A buggy implementation that uses `data[mask].mean(dim=0)` (instead of `[0]`) returns the class CENTROID, which trains a different visualization (also useful, but not what ARENA's `HOLDOUT_DATA` represents).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()